# Step 3, Part 3: BB84 as a link in the network design

*Secure the Grid, step 3 · Option A (reuse BB84 from Step 1)*

This notebook runs BB84 as one specific link from the network design: the **3.7 km link between control center CHA 3 and substation CHA 24** in the Chattanooga-area grid (see `3b_network_design.ipynb`). It answers three questions:

1. Does the link produce a matching key when nobody is listening?
2. With realistic line noise, where should the alarm be set?
3. How large a tap does that alarm catch?

**Line noise.** EPB's field test measured QBER of 1.4% to 3.9% on links of 8 to 10 km (Evans et al., IEEE Access 2021). This notebook uses 2.5% as the link's baseline noise, an assumption inside that range.

**The eavesdropper inside one circuit.** Step 1 modeled Eve with a separate measurement per qubit. Here each tapped round gets an extra "Eve" qubit: Eve rotates into her random basis, copies the qubit's value onto her qubit with a CNOT, and rotates back. For Bob this is identical to intercept-resend, and it lets a whole batch of rounds, tapped or not, run as **one circuit**, so it can be sent to qBraid as a single job.

> **Kernel:** pick **qBraid-SDK (v0.12.2)** in the Select Kernel box.

In [ ]:
import importlib.util
missing = [m for m in ("qbraid", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Wrong kernel: pick qBraid-SDK (v0.12.2) at the top right, then run this cell again.")
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError

LINK = "CHA 3 (control center) to CHA 24, 3.7 km"
LINE_NOISE = 0.025      # baseline QBER of the fiber (assumption inside EPB's measured 1.4-3.9%)
ROUNDS = 12             # BB84 rounds per circuit; with 12 Eve qubits this is 24 qubits, within qBraid's free simulators
SHOTS = 200             # shots per batch for the qBraid runs
WINDOW = 10             # the alarm decides on a window of 10 batches (120 rounds, about 60 after sifting)

noise = NoiseModel()
noise.add_all_qubit_readout_error(ReadoutError([[1 - LINE_NOISE, LINE_NOISE], [LINE_NOISE, 1 - LINE_NOISE]]))
clean_sim = AerSimulator(method="stabilizer")                     # BB84 uses only Clifford gates, so this is fast
line_sim = AerSimulator(method="stabilizer", noise_model=noise)   # the same, with the link's line noise
rng = np.random.default_rng(7)
print("Setup OK. Link:", LINK)

## 1. Build one batch of rounds

Each batch draws new random bits and bases for Alice and Bob. `tap_fraction` sets the share of rounds Eve intercepts.

In [ ]:
def bb84_batch(tap_fraction=0.0, rounds=ROUNDS):
    bits, a_bases, b_bases = (rng.integers(0, 2, rounds) for _ in range(3))
    tapped = rng.random(rounds) < tap_fraction
    e_bases = rng.integers(0, 2, rounds)
    qc = QuantumCircuit(2 * rounds, rounds)       # qubits 0..rounds-1: the link; rounds..2*rounds-1: Eve's copies
    for q in range(rounds):
        if bits[q]: qc.x(q)                       # Alice encodes her bit...
        if a_bases[q]: qc.h(q)                    # ...in her basis
        if tapped[q]:                             # Eve measures in her own random basis (as a CNOT copy)
            if e_bases[q]: qc.h(q)
            qc.cx(q, rounds + q)
            if e_bases[q]: qc.h(q)
        if b_bases[q]: qc.h(q)                    # Bob measures in his basis
    qc.measure(range(rounds), range(rounds))
    return qc, (bits, a_bases, b_bases)

def errors_from_counts(counts, key):
    # errors and total on the sifted rounds (bases matched), over every shot
    bits, a_bases, b_bases = key
    keep = [q for q in range(len(bits)) if a_bases[q] == b_bases[q]]
    errors = total = 0
    for bitstring, c in counts.items():
        bob = bitstring.replace(" ", "").zfill(len(bits))[::-1]   # qubit 0 is the last character
        for q in keep:
            errors += c * (int(bob[q]) != bits[q]); total += c
    return errors, total

def qber_from_counts(counts, key):
    e, t = errors_from_counts(counts, key)
    return e / t if t else float("nan")

def run_window(sim, tap_fraction=0.0, batches=WINDOW, shots=50):
    # QBER pooled over a window of batches; each batch has new random bits, bases and tapped rounds.
    # Aer's stabilizer method runs h, x, cx and measure directly, so no transpile is needed.
    E = T = 0
    for _ in range(batches):
        qc, key = bb84_batch(tap_fraction)
        e, t = errors_from_counts(sim.run(qc, shots=shots).result().get_counts(), key)
        E += e; T += t
    return E / T

print("Quiet link, no line noise:", round(run_window(clean_sim), 3))
print("Quiet link, with line noise:", round(run_window(line_sim), 3))
print("Full tap, with line noise:", round(run_window(line_sim, 1.0), 3))

## 2. Calibrate the alarm on the quiet link

One batch keeps only about 6 rounds after sifting, too few to judge. The alarm instead decides on a **window of 10 batches** (about 60 sifted rounds). Run many quiet windows; the alarm level is their mean QBER plus three standard deviations, so a quiet link rarely raises a false alarm.

In [ ]:
quiet = np.array([run_window(line_sim) for _ in range(100)])
alarm = quiet.mean() + 3 * quiet.std()
print(f"Quiet-link QBER: mean {quiet.mean():.3f}, std {quiet.std():.3f}")
print(f"Alarm level: {alarm:.3f}   (BB84 can no longer make a secure key above ~0.11)")

## 3. How large a tap does the alarm catch?

For each tap fraction, run windows and count how often the QBER crosses the alarm level.

In [ ]:
fractions = np.array([0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.75, 1.0])
detect, mean_qber = [], []
for f in fractions:
    q = np.array([run_window(line_sim, f) for _ in range(60)])
    mean_qber.append(q.mean()); detect.append(np.mean(q > alarm))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(fractions, mean_qber, "o-", label="mean QBER")
ax[0].axhline(alarm, color="tab:red", ls="--", label=f"alarm ({alarm:.3f})")
ax[0].axhline(0.11, color="gray", ls=":", label="BB84 key limit (~0.11)")
ax[0].set_xlabel("fraction of rounds Eve taps"); ax[0].set_ylabel("QBER"); ax[0].legend(); ax[0].set_title(LINK)
ax[1].plot(fractions, detect, "o-", color="tab:red")
ax[1].set_xlabel("fraction of rounds Eve taps"); ax[1].set_ylabel("share of windows that raise the alarm")
ax[1].set_ylim(-0.05, 1.05); ax[1].set_title(f"Detection per window of {WINDOW} batches")
plt.tight_layout(); plt.savefig("3a_link_alarm.png", dpi=150); plt.show()

caught = [f for f, d in zip(fractions, detect) if d >= 0.95]
print("Smallest tap caught in at least 95% of windows:", caught[0] if caught else "none")
for f, m, d in zip(fractions, mean_qber, detect):
    print(f"tap {f:.2f}: mean QBER {m:.3f}, alarm rate {d:.2f}")

## 4. Run the link through qBraid

The same circuit runs on qBraid's free backends: one quiet batch and one fully tapped batch. Both are free. Copy each job ID into the run record in `WRITEUP.md`.

- `qbraid:qbraid:sim:qir-sv` is noise-free, so the quiet batch should show 0% QBER.
- `ionq:ionq:sim:simulator` with `noise_model` adds IonQ's gate noise. The tapped batch uses CNOTs for Eve's copies, so this noise shows up there.

In [ ]:
from qbraid.runtime import QbraidProvider
import warnings
warnings.filterwarnings("ignore", message="Circuit contains measurement gates")
provider = QbraidProvider()

runs = {}
for label, f in [("quiet", 0.0), ("tapped", 1.0)]:
    qc, key = bb84_batch(f)
    for dev_id, opts in [("qbraid:qbraid:sim:qir-sv", {}), ("ionq:ionq:sim:simulator", {"runtime_options": {"noise_model": "aria-1"}})]:
        job = provider.get_device(dev_id).run(qc, shots=SHOTS, **opts)
        counts = job.result().data.get_counts()
        runs[(label, dev_id)] = (job.id, qber_from_counts(counts, key))
        print(f"{label:6s} {dev_id:28s} job {job.id}  QBER {runs[(label, dev_id)][1]:.3f}")

## What to have ready (Step 3, Part 3)

- The quiet-link QBER and the alarm level from section 2.
- The smallest tap caught in 95% of windows, and the plot `3a_link_alarm.png`, from section 3.
- The qBraid job IDs and QBERs from section 4, in the run record.

**Limits to state:** line noise is modeled as a bit flip at Bob's detector, not as fiber loss, and 2.5% is an assumption within EPB's measured range. The alarm judges windows of about 60 sifted rounds; real systems use far longer windows, which would catch smaller taps.